In [2]:
import os

# --- CONFIGURACIÓN DE RUTAS PARA NOTEBOOK ---
# Pega aquí la ruta de tu carpeta 'data'
data_dir = r"\luisf\Documents\trabajo-California-Traffic-Collision\data"
delta_path = os.path.join(data_dir, "delta_lake")

print(f"Buscando Delta Lake en: {delta_path}")

Buscando Delta Lake en: \luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake


In [3]:
import pandas as pd
import os

# --- 1. RUTA A LA CARPETA QUE YA TIENE LA VERSIÓN 2 ---
# (Asegúrate de que la carpeta 'collisions' esté en esta ruta)
ruta_base = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\raw"

# --- 2. LEER TODOS LOS ARCHIVOS PARQUET ---
print("Leyendo archivos de la Versión 2...")

# Delta Lake guarda los datos en varios archivos .parquet, los buscamos todos:
archivos_parquet = [f for f in os.listdir(ruta_base) if f.endswith('.parquet')]

if not archivos_parquet:
    print("Error: No encontré archivos .parquet. Revisa que la ruta sea correcta.")
else:
    # Leemos y concatenamos todo en un solo DataFrame de Pandas
    lista_df = [pd.read_parquet(os.path.join(ruta_base, f)) for f in archivos_parquet]
    df_final = pd.concat(lista_df, ignore_index=True)

    print(f"¡LOGRADO! Se cargaron {len(df_final)} registros exitosamente.")
    
    # --- 3. VISTA PREVIA ---
    print("\nResumen de las primeras filas:")
    print(df_final.head())

Leyendo archivos de la Versión 2...
¡LOGRADO! Se cargaron 400000 registros exitosamente.

Resumen de las primeras filas:
   case_id db_year  jurisdiction officer_id reporting_district chp_shift  \
0  0081715    2021           NaN        NaN                NaN       NaN   
1  0726202    2021           NaN        NaN                NaN       NaN   
2  3858022    2021           NaN        NaN                NaN       NaN   
3  3899441    2021           NaN        NaN                NaN       NaN   
4  3899442    2021           NaN        NaN                NaN       NaN   

  population county_city_location county_location special_condition  ...  \
0        NaN                  NaN             NaN               NaN  ...   
1        NaN                  NaN             NaN               NaN  ...   
2        NaN                  NaN             NaN               NaN  ...   
3        NaN                  NaN             NaN               NaN  ...   
4        NaN                  NaN         

In [4]:
import pandas as pd
import os

# --- 1. ASEGÚRATE QUE ESTA RUTA SEA EXACTA ---
# Si tu carpeta en el disco C se llama diferente, cámbialo aquí.
# Ejemplo: si es "C:\Data\delta_lake", pon esa.
base_path = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake" 

def cargar_tabla(nombre_carpeta):
    ruta = os.path.join(base_path, nombre_carpeta)
    # Verificamos si la carpeta existe antes de intentar leer
    if not os.path.exists(ruta):
        print(f"❌ ERROR: No existe la carpeta: {ruta}")
        return pd.DataFrame()
    
    # Buscamos archivos .parquet
    archivos = [f for f in os.listdir(ruta) if f.endswith('.parquet')]
    if not archivos:
        print(f"⚠️ ADVERTENCIA: No hay archivos .parquet en: {ruta}")
        return pd.DataFrame()
        
    return pd.concat([pd.read_parquet(os.path.join(ruta, f)) for f in archivos], ignore_index=True)

# Cargamos las tablas
print("Cargando datos...")
collisions = cargar_tabla("collisions")
victims_clean = cargar_tabla("involved_victims")
parties = cargar_tabla("parties")
victims_raw = cargar_tabla("victims")

# --- 2. ESTO DEBE MOSTRAR NÚMEROS MAYORES A 0 ---
if not collisions.empty:
    print(f"✅ ¡ÉXITO! Colisiones cargadas: {len(collisions)} filas")
    print("\nColumnas disponibles en Colisiones:", collisions.columns.tolist())
else:
    print("❌ Las tablas siguen vacías. Revisa la ruta 'base_path' arriba.")

Cargando datos...
✅ ¡ÉXITO! Colisiones cargadas: 600000 filas

Columnas disponibles en Colisiones: ['case_id', 'jurisdiction', 'officer_id', 'reporting_district', 'chp_shift', 'population', 'county_city_location', 'county_location', 'special_condition', 'beat_type', 'chp_beat_type', 'city_division_lapd', 'chp_beat_class', 'beat_number', 'primary_road', 'secondary_road', 'distance', 'direction', 'intersection', 'weather_1', 'weather_2', 'state_highway_indicator', 'caltrans_county', 'caltrans_district', 'state_route', 'route_suffix', 'postmile_prefix', 'postmile', 'location_type', 'ramp_intersection', 'side_of_highway', 'tow_away', 'collision_severity', 'killed_victims', 'injured_victims', 'party_count', 'primary_collision_factor', 'pcf_violation_category', 'pcf_violation', 'pcf_violation_subsection', 'hit_and_run', 'type_of_collision', 'motor_vehicle_involved_with', 'pedestrian_action', 'road_surface', 'road_condition_1', 'road_condition_2', 'lighting', 'control_device', 'chp_road_type'

In [27]:
# SECCIÓN: CONSULTAS DE POBLACIÓN Y RIESGO 

# Q3. Colisiones ocurridas bajo condiciones climáticas adversas (No despejado)
# Filtramos todo lo que NO sea 'clear'
q3 = collisions[collisions['weather_1'] != 'clear']
print(f"Q3: Colisiones bajo condiciones climaticas adversas: {len(q3)} filas")

# Q4. Víctimas que no portaban equipo de seguridad (cinturón, casco, etc.)
# Usamos 'victims_raw' para detalles específicos del equipo
q4 = victims_raw[victims_raw['victim_safety_equipment_1'].str.contains('none', na=False, case=False)]
print(f"Q4: Victimas sin equipo de seguridad: {len(q4)} filas")

# Q5. Accidentes ocurridos en condiciones de oscuridad (con o sin alumbrado)
q5 = collisions[collisions['lighting'].str.contains('dark', na=False, case=False)]
print(f"Q5: Colisiones en condiciones de oscuridad: {len(q5)} filas")

# Q6. Colisiones que tuvieron lugar en intersecciones
q6 = collisions[collisions['location_type'] == 'intersection']
print(f"Q6: Colisiones en intersecciones: {len(q6)} filas")

# Q7. Perfil de riesgo: Mujeres de la tercera edad (> 65 años)
# IMPORTANTE: Usamos 'victims_clean' (tu tabla sin repetidos) para estadística real
q7 = victims_clean[(victims_clean['victim_sex'] == 'female') & (victims_clean['victim_age'] > 65)]
print(f"Q7: Víctimas de la tercera edad: {len(q7)} filas")

# Q8. Accidentes donde estuvieron involucradas motocicletas
q8 = parties[parties['statewide_vehicle_type'] == 'motorcycle or scooter']
print(f"Q8: Accidentes con motocicletas o scooters: {len(q8)} filas")

# Q9. Colisiones causadas por exceso de velocidad (PCF Violation)
q9 = collisions[collisions['pcf_violation_category'] == 'speeding']
print(f"Q9: Colisiones por exceso de velocidad: {len(q9)} filas")

# Q10. Víctimas que fueron expulsadas del vehículo durante el impacto
q10 = victims_raw[victims_raw['victim_ejected'] == 'fully ejected']
print(f"Q10: Víctimas expulsadas del vehículo: {len(q10)} filas")

Q3: Colisiones bajo condiciones climaticas adversas: 172026 filas
Q4: Victimas sin equipo de seguridad: 5112 filas
Q5: Colisiones en condiciones de oscuridad: 196716 filas
Q6: Colisiones en intersecciones: 11874 filas
Q7: Víctimas de la tercera edad: 3138 filas
Q8: Accidentes con motocicletas o scooters: 7158 filas
Q9: Colisiones por exceso de velocidad: 187440 filas
Q10: Víctimas expulsadas del vehículo: 15774 filas


In [13]:
# SECCIÓN: ANÁLISIS ESTADÍSTICOS ACTUARIALES 

# A1. Letalidad promedio según rangos de edad (Uso de tabla Limpia)
# Cruzamos víctimas con colisiones para obtener el conteo de fallecidos
letalidad_df = pd.merge(victims_clean, collisions[['case_id', 'killed_victims']], on='case_id')
bins_edad = [0, 25, 60, 100]
etiquetas = ['Jóvenes', 'Adultos', 'Ancianos']
print("\nA1. Promedio de fallecidos por rango de edad:")
print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())

# A2. Relación entre uso de celular y culpabilidad en el accidente
parties['at_fault'] = pd.to_numeric(parties['at_fault'], errors='coerce')
print("\nA2. Probabilidad de ser culpable según uso de celular:")
print(parties.groupby('cellphone_in_use')['at_fault'].mean())

# A3. Top 5 de marcas de vehículos con mayor frecuencia de culpabilidad
print("\nA3. Top 5 marcas de vehículos involucradas en choques por culpa:")
marcas_culpa = parties[parties['at_fault'] == 1]['vehicle_make'].value_counts().head(5)
print(marcas_culpa)

# A4. Severidad del accidente según el estado de la superficie de la vía
print("\nA4. Promedio de heridos según tipo de carretera (Road Surface):")
print(collisions.groupby('road_surface')['injured_victims'].mean())

# A5. Matriz de Sobriedad vs Severidad del Accidente
# Este análisis es fundamental para medir el impacto del alcohol en la gravedad
sobriedad_analisis = pd.merge(parties[['case_id', 'party_sobriety']], 
                              collisions[['case_id', 'collision_severity']], on='case_id')
print("\nA5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque")
print(pd.crosstab(sobriedad_analisis['party_sobriety'], sobriedad_analisis['collision_severity']))


A1. Promedio de fallecidos por rango de edad:


C:\Users\luisf\AppData\Local\Temp\ipykernel_13068\624764559.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())


victim_age
Jóvenes     0.012998
Adultos     0.019462
Ancianos    0.026709
Name: killed_victims, dtype: float64

A2. Probabilidad de ser culpable según uso de celular:
cellphone_in_use
0.0    0.477592
1.0    0.536195
Name: at_fault, dtype: float64

A3. Top 5 marcas de vehículos involucradas en choques por culpa:
vehicle_make
toyota       40710
ford         38166
honda        28080
chevrolet    26178
nissan       15570
Name: count, dtype: int64

A4. Promedio de heridos según tipo de carretera (Road Surface):
road_surface
dry         0.539094
slippery    0.571429
snowy       0.444837
wet         0.473858
Name: injured_victims, dtype: float64

A5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque
collision_severity                      fatal  other injury    pain  \
party_sobriety                                                        
had been drinking, impairment unknown     396          3456    4032   
had been drinking, not under influence    612          3708    5724   
had be

In [13]:
# --- TENDENCIAS Y VALORES ATÍPICOS (DIAGNÓSTICO) ---

print("--- 1. Identificación de Horas Pico de Accidentes (Tendencia Temporal) ---")
# ¿A qué hora exacta ocurren más choques?
print(collisions['collision_time'].value_counts().head(10))

print("\n--- 2. Valores Atípicos: Accidentes con número inusual de víctimas ---")
# Buscamos accidentes que se salen de lo normal (ejemplo: más de 10 víctimas)
outliers_victimas = collisions[collisions['injured_victims'] > 10]
print(f"Accidentes con >10 heridos: {len(outliers_victimas)}")

print("\n--- 3. Tendencia: Tipo de Objeto más chocado en colisiones de un solo vehículo ---")
# Ayuda a entender si el riesgo es por infraestructura o por otros conductores
print(collisions['type_of_collision'].value_counts())



--- 1. Identificación de Horas Pico de Accidentes (Tendencia Temporal) ---
collision_time
15:00:00    3972
16:00:00    3972
08:00:00    3840
15:30:00    3660
18:00:00    3474
17:00:00    3414
16:30:00    3378
17:30:00    3264
18:30:00    3252
14:30:00    3156
Name: count, dtype: int64

--- 2. Valores Atípicos: Accidentes con número inusual de víctimas ---
Accidentes con >10 heridos: 18

--- 3. Tendencia: Tipo de Objeto más chocado en colisiones de un solo vehículo ---
type_of_collision
rear end      192888
broadside     119634
hit object    106998
sideswipe     100488
head-on        26088
pedestrian     17112
overturned     16896
other          14586
Name: count, dtype: int64


In [ ]:
print("\n--- 4. Análisis de Consistencia: Víctimas sin edad registrada (Missing Data) ---")
# Los valores nulos o 0 en edad pueden arruinar un cálculo actuarial
print(f"Víctimas con edad 0 o nula: {len(victims_clean[victims_clean['victim_age'] <= 0])}")

print("\n--- 5. Outliers de Edad: Conductores 'Super-Senior' ---")
# ¿Hay registros de conductores con edades poco creíbles (ej. > 100 años)?
print(parties[parties['party_age'] > 95][['party_age', 'vehicle_make']])


print("\n--- 7. Anomalía: Choques con severidad alta pero sin heridos registrados ---")
# Esto detecta errores de captura en el reporte policial
anomalia_reporte = collisions[(collisions['collision_severity'] == 'fatal') & (collisions['killed_victims'] == 0)]
print(f"Registros inconsistentes (Fatal pero 0 muertos): {len(anomalia_reporte)}")

print("\n--- 8. Tendencia: Relación entre Antigüedad del Vehículo y Severidad ---")
# ¿Los autos más viejos tienen accidentes más graves?
parties['vehicle_year'] = pd.to_numeric(parties['vehicle_year'], errors='coerce')
print(parties.groupby(pd.cut(parties['vehicle_year'], bins=[1900, 2000, 2010, 2025]))['at_fault'].count())




--- 4. Análisis de Consistencia: Víctimas sin edad registrada (Missing Data) ---
Víctimas con edad 0 o nula: 816

--- 5. Outliers de Edad: Conductores 'Super-Senior' ---
        party_age   vehicle_make
4660        103.0      chevrolet
5236         96.0           None
9604         97.0      chevrolet
15201        98.0         toyota
15302       101.0  mercedes-benz
...           ...            ...
572125      104.0           None
573795       96.0       kenworth
591340       96.0   freightliner
594954       96.0           ford
595303      100.0          honda

[100 rows x 2 columns]

--- 7. Anomalía: Choques con severidad alta pero sin heridos registrados ---
Registros inconsistentes (Fatal pero 0 muertos): 0

--- 8. Tendencia: Relación entre Antigüedad del Vehículo y Severidad ---
vehicle_year
(1900, 2000]    235584
(2000, 2010]    307050
(2010, 2025]         6
Name: at_fault, dtype: int64

--- 9. Distribución del tipo de herida en la tabla limpia ---
victim_degree_of_injury
no injur

C:\Users\luisf\AppData\Local\Temp\ipykernel_2604\2617019949.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(parties.groupby(pd.cut(parties['vehicle_year'], bins=[1900, 2000, 2010, 2025]))['at_fault'].count())


In [16]:
print("\n--- 9. Distribución del tipo de herida en la tabla limpia ---")
print(victims_raw['victim_degree_of_injury'].value_counts())




--- 9. Distribución del tipo de herida en la tabla limpia ---
victim_degree_of_injury
no injury               289020
complaint of pain       218268
other visible injury     75582
severe injury            13188
killed                    3942
Name: count, dtype: int64
